# 🎙️ Thai Voice Command: Data Collector Tool

**ยินดีต้อนรับสู่ภารกิจพิเศษ (The Localization Project)!** 
เครื่องมือนี้ถูกสร้างขึ้นมาเพื่อช่วยให้คุณสร้างชุดข้อมูล (Dataset) เสียงพูดภาษาไทยของคุณเองได้อย่างรวดเร็วและเป็นระเบียบ

### 🎯 กฎเหล็กของ Data Engineer (ทบทวนความรู้ Lab 03)
1. **ความหลากหลาย:** ควรชวนเพื่อนหลายๆ คนมาสลับกันพูด เพื่อให้ AI คุ้นเคยกับโทนเสียงที่ต่างกัน
2. **Data Consistency:** เสียงทุกไฟล์ต้องยาว **1 วินาทีเป๊ะ (16,000 Data Points)** ซึ่งเครื่องมือนี้จะจัดการ "ตัดขอบ (Truncate)" และ "เติมความเงียบ (Zero-Padding)" ให้คุณอัตโนมัติ!

---

## 🛠️ Step 1: ตั้งค่าโฟลเดอร์และคำศัพท์เป้าหมาย
รัน Cell ด้านล่างเพื่อเตรียมโครงสร้างโฟลเดอร์ (สามารถแก้ไขคำศัพท์ในตัวแปร `vocab_list` ได้ตามโปรเจกต์ของกลุ่มคุณ)

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from IPython.display import HTML, clear_output, display
from google.colab.output import eval_js
from base64 import b64decode
import shutil
from google.colab import files

# 🎯 กำหนดคำศัพท์ภาษาไทยที่ต้องการ (เปลี่ยนได้ตามใจชอบ)
vocab_list = ['เดิน', 'หยุด', 'ซ้าย', 'ขวา']

# สร้างโฟลเดอร์หลัก
dataset_dir = 'thai_speech_dataset'
os.makedirs(dataset_dir, exist_ok=True)

# สร้างโฟลเดอร์ย่อยตามคำศัพท์
for word in vocab_list:
    os.makedirs(os.path.join(dataset_dir, word), exist_ok=True)

print(f"✅ สร้างโฟลเดอร์สำหรับเก็บคำศัพท์ {vocab_list} เรียบร้อยแล้ว!")
print("พร้อมสำหรับการอัดเสียงใน Step ถัดไป...")

--- 
## 🔴 Step 2: ห้องอัดเสียง (Recording Studio)
**วิธีใช้งาน:** 
1. เลือกคำศัพท์จาก Dropdown Menu ด้านล่าง
2. กดปุ่ม **Run (▶️)** ที่มุมซ้ายของ Cell นี้
3. เมื่อปุ่มไมโครโฟนปรากฏขึ้น ให้กดแล้วพูดคำศัพท์นั้นให้ชัดเจน
4. ทำซ้ำจนกว่าจะได้จำนวนไฟล์ที่ต้องการ (แนะนำอย่างน้อยคำละ 20-30 ไฟล์)

In [ ]:
#@title 🎛️ เลือกคำศัพท์ที่กำลังจะอัด
TARGET_WORD = "\u0E40\u0E14\u0E34\u0E19" #@param ["เดิน", "หยุด", "ซ้าย", "ขวา"]

# --- UI อัดเสียง ---
AUDIO_HTML = """
<script>
var my_div = document.createElement("DIV");
var my_btn = document.createElement("BUTTON");
my_btn.style.padding = "15px 30px"; my_btn.style.fontSize = "20px"; 
my_btn.style.cursor = "pointer"; my_btn.style.backgroundColor = "#d93025";
my_btn.style.color = "white"; my_btn.style.border = "none"; my_btn.style.borderRadius = "8px";
var t = document.createTextNode("🎤 กดเพื่ออัดคำว่า: " + TARGET_WORD_JS);
my_btn.appendChild(t); my_div.appendChild(my_btn); document.body.appendChild(my_div);
var base64data = 0; var reader; var recorder, gumStream; var recordButton = my_btn;
var handleSuccess = function(stream) {
  gumStream = stream;
  recorder = new MediaRecorder(stream, { mimeType : 'audio/webm;codecs=opus' });
  recorder.ondataavailable = function(e) {
    reader = new FileReader(); reader.readAsDataURL(e.data);
    reader.onloadend = function() { base64data = reader.result; }
  };
  recorder.start(); recordButton.innerText = "🔴 กำลังบันทึก... พูดเลย!";
  // อัดเสียง 1.5 วินาที เพื่อให้คลุมเสียงพูดพอดี
  setTimeout(() => { recorder.stop(); gumStream.getAudioTracks()[0].stop(); recordButton.innerText = "✅ สำเร็จ! กำลังประมวลผล..."; }, 1500);
};
recordButton.onclick = function() { navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess); }
function getAudio() { return new Promise(resolve => { var interval = setInterval(() => { if (base64data != 0) { clearInterval(interval); resolve(base64data); } }, 100); }); }
</script>
"""
# ส่งค่าคำศัพท์ไปให้ JS แสดงบนปุ่ม
display(HTML(f"<script>var TARGET_WORD_JS = '{TARGET_WORD}';</script>"))
display(HTML(AUDIO_HTML))
data = eval_js("getAudio()")
binary = b64decode(data.split(',')[1])

# เซฟไฟล์ชั่วคราวและแปลงเป็น wav
temp_webm = 'temp.webm'
temp_wav = 'temp.wav'
with open(temp_webm, 'wb') as f: f.write(binary)
!ffmpeg -y -i {temp_webm} -ac 1 -ar 16000 {temp_wav} -loglevel quiet

# --- Data Consistency Engine (ปรับขนาดไฟล์) ---
y, sr = librosa.load(temp_wav, sr=16000)
expected_length = 16000 # 1 วินาที

if len(y) > expected_length:
    y = y[:expected_length] # ตัดส่วนที่เกิน (Truncating)
elif len(y) < expected_length:
    y = np.pad(y, (0, expected_length - len(y)), 'constant') # เติม 0 (Padding)

# --- บันทึกไฟล์ลงโฟลเดอร์ที่ถูกต้อง ---
filename = f"{TARGET_WORD}_{int(time.time())}.wav"
save_path = os.path.join(dataset_dir, TARGET_WORD, filename)
sf.write(save_path, y, sr)

clear_output()
print("===========================================")
print(f"✅ บันทึกไฟล์สำเร็จ: {save_path}")
print(f"📏 ขนาดข้อมูล (Data Points): {len(y)} จุด")
print("===========================================\n")

# วาดกราฟเสียงให้ดูเป็น Feedback
plt.figure(figsize=(10, 2))
plt.plot(y, color='#1a73e8')
plt.title(f'Waveform of "{TARGET_WORD}"')
plt.xlim(0, 16000)
plt.axis('off')
plt.show()

!rm -f {temp_webm} {temp_wav}

--- 
## 📊 Step 3: สรุปยอดข้อมูล (Data Explorer)
ตรวจสอบว่าเราเก็บข้อมูลแต่ละคำศัพท์มาได้เยอะแค่ไหนแล้ว รัน Cell นี้เพื่อดูสรุป

In [ ]:
print("--- 📁 สรุปจำนวนไฟล์ใน Dataset ---")
total_files = 0
for word in vocab_list:
    word_dir = os.path.join(dataset_dir, word)
    if os.path.exists(word_dir):
        count = len(os.listdir(word_dir))
        total_files += count
        print(f"🗣️ {word}: {count} ไฟล์")
        
print("--------------------------------")
print(f"🎉 รวมทั้งหมด: {total_files} ไฟล์")

--- 
## 📥 Step 4: ดาวน์โหลด Dataset กลับสู่ฐานทัพ
เมื่อเก็บข้อมูลจนพอใจแล้ว ให้รัน Cell นี้เพื่อบีบอัดไฟล์ทั้งหมดเป็น `.zip` และดาวน์โหลดลงเครื่องคอมพิวเตอร์ของคุณ เพื่อเตรียมนำไปใช้ฝึกสมอง AI ในขั้นตอนต่อไป!

In [ ]:
zip_name = 'thai_speech_dataset'
shutil.make_archive(zip_name, 'zip', dataset_dir)

print(f"✅ บีบอัดไฟล์ {zip_name}.zip สำเร็จ! กำลังเตรียมดาวน์โหลด...")
files.download(f"{zip_name}.zip")